# Rotten Fruit Detector — SageMaker Studio Training (SSD EfficientDet D5)

Trains **SSD EfficientDet D5** with the TF Object Detection API (`src/dataset.py` + `src/model.py` + `src/train.py`). `main()` builds the label map + TFRecords from the raw Roboflow CSV export, configures the pipeline, then trains, evaluates and exports.

Run from inside the cloned repo. Expected raw dataset:

```text
dataset/
  train/_annotations.csv + *.jpg
  valid/_annotations.csv + *.jpg
  test/_annotations.csv  + *.jpg
```


In [ ]:
# 1. Install system + python deps
!apt-get -qq install -y protobuf-compiler
%pip install -q -r ../requirements.txt

In [ ]:
# 2. Download the raw dataset from S3 and unzip it
import zipfile
from pathlib import Path

import boto3

S3_BUCKET = "YOUR_BUCKET"
S3_KEY = "datasets/dataset.zip"
PROJECT_DIR = Path.cwd().parent

zip_path = PROJECT_DIR / "dataset.zip"
boto3.client("s3").download_file(S3_BUCKET, S3_KEY, str(zip_path))

with zipfile.ZipFile(zip_path) as archive:
    archive.extractall(PROJECT_DIR)

print(sorted(p.name for p in (PROJECT_DIR / "dataset").iterdir()))

In [ ]:
# 3. Prepare TFRecords, configure EfficientDet D5, then train -> evaluate -> export
#    (first run clones the Object Detection API and downloads the pretrained model)
import os
import sys

os.chdir(PROJECT_DIR)
sys.path.insert(0, str(PROJECT_DIR / "src"))

from train import main

main(num_steps=25000, batch_size=2)

In [ ]:
# 4. Upload the exported model back to S3
import sagemaker

session = sagemaker.Session()
uri = session.upload_data(
    path=str(PROJECT_DIR / "exported-models" / "efficientdet_d5"),
    bucket=session.default_bucket(),
    key_prefix="rotten-fruit-detection/exported-model",
)
print("Uploaded to:", uri)